In [ ]:
!pip install plotly pandas requests


In [6]:
import pandas as pd
import requests

url = "https://raw.githubusercontent.com/alura-cursos/challenge2-data-science-LATAM/main/TelecomX_Data.json"
response = requests.get(url)
df = pd.json_normalize(response.json())
df.head()


,customerID,Churn,customer.gender,customer.SeniorCitizen,customer.Partner,customer.Dependents,customer.tenure,phone.PhoneService,phone.MultipleLines,internet.InternetService,...,internet.OnlineBackup,internet.DeviceProtection,internet.TechSupport,internet.StreamingTV,internet.StreamingMovies,account.Contract,account.PaperlessBilling,account.PaymentMethod,account.Charges.Monthly,account.Charges.Total
0,0002-ORFBO,No,Female,0,Yes,Yes,9,Yes,No,DSL,...,Yes,No,Yes,Yes,No,One year,Yes,Mailed check,65.6,593.3
1,0003-MKNFE,No,Male,0,No,No,9,Yes,Yes,DSL,...,No,No,No,No,Yes,Month-to-month,No,Mailed check,59.9,542.4
2,0004-TLHLJ,Yes,Male,0,No,No,4,Yes,No,Fiber optic,...,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,73.9,280.85
3,0011-IGKFF,Yes,Male,1,Yes,No,13,Yes,No,Fiber optic,...,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,98.0,1237.85
4,0013-EXCHZ,Yes,Female,1,Yes,No,3,Yes,No,Fiber optic,...,No,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,83.9,267.4


In [7]:
df.columns = [col.split('.')[-1] for col in df.columns]
df['Total'] = pd.to_numeric(df['Total'], errors='coerce').fillna(0)
df['Monthly'] = pd.to_numeric(df['Monthly'], errors='coerce')
df.drop_duplicates(inplace=True)

for col in df.columns:
    if df[col].isnull().any():
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)

df.to_csv('TelecomX_Cleaned.csv', index=False)
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7267 non-null   str    
 1   Churn             7267 non-null   str    
 2   gender            7267 non-null   str    
 3   SeniorCitizen     7267 non-null   int64  
 4   Partner           7267 non-null   str    
 5   Dependents        7267 non-null   str    
 6   tenure            7267 non-null   int64  
 7   PhoneService      7267 non-null   str    
 8   MultipleLines     7267 non-null   str    
 9   InternetService   7267 non-null   str    
 10  OnlineSecurity    7267 non-null   str    
 11  OnlineBackup      7267 non-null   str    
 12  DeviceProtection  7267 non-null   str    
 13  TechSupport       7267 non-null   str    
 14  StreamingTV       7267 non-null   str    
 15  StreamingMovies   7267 non-null   str    
 16  Contract          7267 non-null   str    
 17  Paperl

In [12]:
import plotly.express as px

fig = px.pie(df, names='Churn', title='Distribución de Abandono (Churn)')
fig.show()


ModuleNotFoundError: No module named 'plotly'

In [ ]:
fig = px.histogram(df, x='Contract', color='Churn', barmode='group', title='Churn por Tipo de Contrato')
fig.show()


In [ ]:
df['Churn_Num'] = df['Churn'].map({'Yes': 1, 'No': 0})
cols = ['tenure', 'Monthly', 'Total', 'Churn_Num']

df_corr = df[cols].corr()
fig = px.imshow(df_corr, text_auto=".2f", aspect="auto", color_continuous_scale='RdBu_r', title='Matriz de Correlación')
fig.show()


# 📄 Informe Final: Conclusiones sobre la Evasión de Clientes (Churn)

Basado en el Análisis Exploratorio de Datos (EDA) realizado, hemos identificado los siguientes patrones principales que explican el alto índice de evasión:

1. **Tipo de Contrato:** Existe una concentración significativa de abandono en los clientes que poseen **contratos de mes a mes (Month-to-month)**. La falta de un compromiso a largo plazo facilita que estos clientes cancelen el servicio ante cualquier insatisfacción o mejor oferta de la competencia.
2. **Cargos Mensuales:** Se observa una correlación positiva moderada entre los cargos mensuales (`Monthly`) y el Churn. Los clientes que pagan tarifas mensuales más altas tienden a abandonar el servicio con mayor frecuencia, lo que sugiere una posible insatisfacción con el balance precio/valor.
3. **Antigüedad (Tenure):** La matriz de correlación muestra una relación negativa profunda entre la antigüedad del cliente (`tenure`) y el Churn. Los clientes más nuevos son mucho más propensos a irse, mientras que los clientes consolidados son más leales.

**Recomendaciones para el equipo de negocio:**
- Implementar incentivos o descuentos para migrar a los clientes de contratos mensuales a contratos anuales o bianuales.
- Crear programas de retención temprana enfocados específicamente en los primeros meses del ciclo de vida del cliente.
- Revisar la estructura de precios de los planes más costosos para asegurar que el valor percibido se alinee con el costo.